# Notebook 02 - Publication Data Clean

This is the second notebook to be run. Here you will find dataframes with informations about the stories such as the number of magazines in which the stories were published,year of writing, year of publication, collaborations and form of the story. Aditionally, we create a dataframe for entities that appear in the texts. Here we will clean and organize these results. 

# 0. Setup

In [45]:
# Libraries

# General
import numpy as np                      # imports the Numpy library for numerical tools
import pandas as pd                     # imports the Pandas library for data manipulation and analysis
import re

# Plotting Options
from matplotlib import pyplot as plt                           # imports the Pyplot module from the Matplotlib library for plotting
import seaborn as sns                                          # imports seaborn for plotting 
plt.rcParams['text.usetex'] = True                             # enables LaTeX rendering for text in plots
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'  # LaTeX preamble

# File management
from pathlib import Path                  # imports path tools from Pathlib for working with file paths
import os                                 # imports the OS library for interacting with the operating system

# defines a function to find the project root directory by looking for a specific marker (default is 'data')
def find_project_root(start=Path.cwd(), marker='data'): 
    current = start
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root() 
os.chdir(PROJECT_ROOT) # go to the root of the project 

# statiscal analysis
from scipy import stats

# tools for text processing
from tools import text_tools
from tools import dataframe_tools as dftools

In [46]:
biblio = text_tools.bibliography()  # defines a bibliography object
print( biblio.files )               # lists the files in the directory

['dreams_in_the_witch.txt', 'ex_oblivione.txt', 'poetry_of_gods.txt', 'doorstep.txt', 'colour_out_of_space.txt', 'iranon.txt', 'street.txt', 'arthur_jermyn.txt', 'crawling_chaos.txt', 'ulthar.txt', 'azathoth.txt', 'shadow_out_of_time.txt', 'haunter.txt', 'martins_beach.txt', 'tree.txt', 'whisperer.txt', 'moon_bog.txt', 'unnamable.txt', 'beast.txt', 'alchemist.txt', 'terrible_old_man.txt', 'nameless.txt', 'reanimator.txt', 'celephais.txt', 'gates_of_silver_key.txt', 'medusas_coil.txt', 'dagon.txt', 'picture_house.txt', 'memory.txt', 'white_ship.txt', 'erich_zann.txt', 'tomb.txt', 'temple.txt', 'he.txt', 'pickman.txt', 'silver_key.txt', 'clergyman.txt', 'old_folk.txt', 'from_beyond.txt', 'descendent.txt', 'nyarlathotep.txt', 'vault.txt', 'hypnos.txt', 'polaris.txt', 'book.txt', 'shunned_house.txt', 'pharoahs.txt', 'randolph_carter.txt', 'innsmouth.txt', 'charles_dexter_ward.txt', 'dunwich.txt', 'festival.txt', 'other_gods.txt', 'redhook.txt', 'hound.txt', 'outsider.txt', 'mountains_of_ma

# 1. Importing and formating dataframe

From [Wikipedia](https://en.wikipedia.org/wiki/H._P._Lovecraft_bibliography), we have generated the following tables

In [47]:
df_fiction = pd.read_csv('data/bronze/dataframes/lovecraft_fiction.csv')
df_collab = pd.read_csv('data/bronze/dataframes/lovecraft_collaborations.csv')

In [48]:
df_fiction.head()

,id,Title,Date written,Date published,Form
0,1,The Alchemist,1908,Nov 1916,Short story
1,2,The Tomb,Jun 1917,Mar 1922,Short story
2,3,Dagon,Jul 1917,Nov 1919,Short story
3,4,A Reminiscence of Dr. Samuel Johnson,Sum-early Fall 1917,Sep 1917,Short story
4,5,Polaris,Spr-Sum 1918,Dec 1920,Short story


In [49]:
df_collab.head()

,Title,Date written,Date published,Collaborators
0,The Battle that Ended the Century,Jun 1934,Jun 1934,R. H. Barlow
1,Bothon,1930,1946,Henry S. Whitehead
2,The Challenge from Beyond,Aug 1935,Sep 1935,C.L. Moore; A. Merritt; Robert E. Howard; Fran...
3,Collapsing Cosmoses,Jun 1935,1938,R. H. Barlow
4,The Crawling Chaos,c. Dec 1920,Apr 1921,Winifred V. Jackson


We also import the $\tt j.son$ file generated from lovecraft bibliography which contains the title of the story, magazine and year in which was published.

In [50]:
df_works = pd.read_json('data/bronze/dataframes/works.json')
df_works = df_works.drop(columns=['full_information'])
df_fiction = pd.read_csv('data/bronze/dataframes/lovecraft_fiction.csv')

cols = [col for col in df_works.columns]
df_works[cols] = df_works[cols].apply(lambda x: x.str.strip().str.lower() if x.dtype == "object" else x)

In [51]:
df_works

,title,magazine,year_published
0,the alchemist.,the united amateur,1916
1,at the mountains of madness.,astounding stories,1936
2,azathoth.,leaves,1938
3,the beast in the cave.,the vagrant,1918
4,beyond the wall of sleep.,pine cones,1919
...,...,...,...
94,"rimel, duane w. ""the tree on the hill.""",polaris,1940
95,"with sterling, kenneth. ""in the walls of eryx.""",weird tales,1939
96,"talman, wilfred blanch. ""two black bottles.""",weird tales,1927
97,"whitehead, henry s. ""bothon.""",amazing stories,1946


There are some stories in which the name of the colaborators where included into the title during the conversion of the .json. We now remove those and create a separate column for colaborations.

In [52]:
collaborators = []
titles = []

for title in df_works['title']:
    if '"' in title:
        collaborator = title.split('"')[0].strip()
        collaborators.append(collaborator)
        titles.append(title.split('"')[1].strip())
    else:
        collaborators.append(None)
        titles.append(title)
        
df_works['collaboration'] = collaborators
df_works['title'] = titles 

We will merge the dataframes $\tt df\_fiction$ and $\tt df\_collab$ with $\tt df\_works$ to add the year in which the stories were written. To do that, we must normalize the story titles, since we will merge with respect to them and drop other columns. We also normalize other columns for the dataframe $\tt df\_works$ which is the one wich we will work the most

In [53]:
cols = ['title', 'collaboration', 'magazine']
df_works[cols] = df_works[cols].apply(lambda col: col.map(dftools.normalize_title))
df_works

,title,magazine,year_published,collaboration
0,the_alchemist,the_united_amateur,1916,None
1,at_the_mountains_of_madness,astounding_stories,1936,None
2,azathoth,leaves,1938,None
3,the_beast_in_the_cave,the_vagrant,1918,None
4,beyond_the_wall_of_sleep,pine_cones,1919,None
...,...,...,...,...
94,the_tree_on_the_hill,polaris,1940,rimel_duane_w
95,in_the_walls_of_eryx,weird_tales,1939,with_sterling_kenneth
96,two_black_bottles,weird_tales,1927,talman_wilfred_blanch
97,bothon,amazing_stories,1946,whitehead_henry_s


In [54]:
df_fiction.rename(columns={'Title': 'title','Form':'form','Date written':'date_written'}, inplace=True)
df_fiction['title']= df_fiction['title'].apply(dftools.normalize_title)
df_fiction = df_fiction.drop(columns={'id', 'Date published'})

In [55]:
df_fiction

,title,date_written,form
0,the_alchemist,1908,Short story
1,the_tomb,Jun 1917,Short story
2,dagon,Jul 1917,Short story
3,a_reminiscence_of_dr_samuel_johnson,Sum-early Fall 1917,Short story
4,polaris,Spr-Sum 1918,Short story
...,...,...,...
60,the_thing_on_the_doorstep,21-24 Aug 1933,Short story
61,the_book,Fragment c. Oct 1933,Fragment
62,the_evil_clergyman,Fall 1933,Letter excerpt
63,the_shadow_out_of_time,10 Nov 1934-22 Feb 1935,Novella


In [56]:
df_fiction[df_fiction["title"].str.contains("madness", case=False, na=False)]

,title,date_written,form
57,at_the_mountains_of_madness,24 Feb-Mar 22 1931,Novella


We did not drop the column form because we will use it later. Doing the same for $\tt df\_collab$ 

In [57]:
df_collab.rename(columns={'Title': 'title','Date written':'date_written'}, inplace=True)
df_collab['title']= df_collab['title'].apply(dftools.normalize_title)
df_collab = df_collab.drop(columns={'Collaborators','Date published'})

In [58]:
df_collab

,title,date_written
0,the_battle_that_ended_the_century,Jun 1934
1,bothon,1930
2,the_challenge_from_beyond,Aug 1935
3,collapsing_cosmoses,Jun 1935
4,the_crawling_chaos,c. Dec 1920
5,the_curse_of_yig,Spring 1928
6,the_diary_of_alonzo_typer,Oct 1935
7,the_disinterment,Sep 1935
8,the_electric_executioner,Jul 1929
9,the_green_meadow,c. 1918–1919


We now create a function that extracts the year from date_written

In [59]:
def extract_year(date_str):
    """
    Extracts a 4-digit year from messy date strings.
    Examples handled:
      - "24 Feb-Mar 22 1931"  → 1931
      - "Sum-early Fall 1917" → 1917
      - "March 1945"          → 1945
      - "1899"                → 1899
      - None / NaN            → None
    """
    if not isinstance(date_str, str):
        return None
    
    # Find all 4-digit numbers that look like years (1800–2099)
    years = re.findall(r'\b(1[89]\d{2}|20\d{2})\b', date_str)
    
    if years:
        return int(years[-1])  # return the last match (most likely the year)
    
    return None

In [60]:
df_fiction['year_written'] = df_fiction['date_written'].apply(extract_year).astype('Int64') 

In [61]:
df_fiction.head()

,title,date_written,form,year_written
0,the_alchemist,1908,Short story,1908
1,the_tomb,Jun 1917,Short story,1917
2,dagon,Jul 1917,Short story,1917
3,a_reminiscence_of_dr_samuel_johnson,Sum-early Fall 1917,Short story,1917
4,polaris,Spr-Sum 1918,Short story,1918


In [62]:
df_collab['year_written'] = df_collab['date_written'].apply(extract_year).astype('Int64') 

In [63]:
df_collab.head()

,title,date_written,year_written
0,the_battle_that_ended_the_century,Jun 1934,1934
1,bothon,1930,1930
2,the_challenge_from_beyond,Aug 1935,1935
3,collapsing_cosmoses,Jun 1935,1935
4,the_crawling_chaos,c. Dec 1920,1920


We merge these two into one

In [64]:
df_collab_fiction = pd.concat([df_collab,df_fiction])
df_collab_fiction

,title,date_written,year_written,form
0,the_battle_that_ended_the_century,Jun 1934,1934,NaN
1,bothon,1930,1930,NaN
2,the_challenge_from_beyond,Aug 1935,1935,NaN
3,collapsing_cosmoses,Jun 1935,1935,NaN
4,the_crawling_chaos,c. Dec 1920,1920,NaN
...,...,...,...,...
60,the_thing_on_the_doorstep,21-24 Aug 1933,1933,Short story
61,the_book,Fragment c. Oct 1933,1933,Fragment
62,the_evil_clergyman,Fall 1933,1933,Letter excerpt
63,the_shadow_out_of_time,10 Nov 1934-22 Feb 1935,1935,Novella


And now we merge with $\tt df\_works$

In [65]:
df = pd.merge(df_works,df_collab_fiction, on = ['title'],how='left')
df['form'] = df['form'].apply(dftools.normalize_title)
df

,title,magazine,year_published,collaboration,date_written,year_written,form
0,the_alchemist,the_united_amateur,1916,None,1908,1908,short_story
1,at_the_mountains_of_madness,astounding_stories,1936,None,24 Feb-Mar 22 1931,1931,novella
2,azathoth,leaves,1938,None,Fragment Jun 1922,1922,fragment
3,the_beast_in_the_cave,the_vagrant,1918,None,NaN,<NA>,None
4,beyond_the_wall_of_sleep,pine_cones,1919,None,Spr 1919,1919,short_story
...,...,...,...,...,...,...,...
94,the_tree_on_the_hill,polaris,1940,rimel_duane_w,May 1934,1934,None
95,in_the_walls_of_eryx,weird_tales,1939,with_sterling_kenneth,Jan 1936,1936,None
96,two_black_bottles,weird_tales,1927,talman_wilfred_blanch,Jun-Oct 1926,1926,None
97,bothon,amazing_stories,1946,whitehead_henry_s,1930,1930,None


The rows with $\tt form = None$ must be added manually. Providing these references to NotebookLM, we got the following table *(that we still need to check)*

In [66]:
df_collab_form = pd.read_json('data/bronze/dataframes/collaboration_story_form.json')
df_collab_form.head()

,title,form
0,the_beast_in_the_cave,short_story
1,poetry_and_the_gods,short_story
2,the_crawling_chaos,short_story
3,herbert_westreanimator_i_from_the_dark,short_story
4,the_horror_at_martins_beach,short_story


Now we merge this with df

In [67]:
df = pd.merge(df,df_collab_form,on='title',how='left',suffixes=('','_y'))
df['form'] = df['form'].combine_first(df['form_y'])
df = df.drop(columns=['form_y'])


In [68]:
df = df.sort_values(by='year_published',ignore_index=True)
df.to_csv('data/gold/lovecraft_works.csv', index=False)

In [69]:
df

,title,magazine,year_published,collaboration,date_written,year_written,form
0,the_alchemist,the_united_amateur,1916,None,1908,1908,short_story
1,a_reminiscence_of_dr_samuel_johnson,the_united_amateur,1917,None,Sum-early Fall 1917,1917,short_story
2,the_beast_in_the_cave,the_vagrant,1918,None,NaN,<NA>,short_story
3,beyond_the_wall_of_sleep,pine_cones,1919,None,Spr 1919,1919,short_story
4,the_white_ship,the_united_amateur,1919,None,c. Oct 1919,1919,short_story
...,...,...,...,...,...,...,...
94,bothon,amazing_stories,1946,whitehead_henry_s,1930,1930,short_story
95,the_dreamquest_of_unknown_kadath,the_arkham_sampler,1948,None,Oct 1926-22 Jan 1927,1927,novella
96,sweet_ermengarde_or_the_heart_of_a_country_girl,mirage,1965,None,NaN,<NA>,short_story
97,ex_oblivione,magazine_of_horror,1968,None,1920 – Mar 1921,1921,short_story


# 2. Generating dataframe for the entities

Last but not least, we create a dataframe including the entities, artifacts and places that appeared in the stories

In [70]:
entities_list = [
    'Azathoth',
    'Nyarlathotep',
    'Yog-Sothoth',
    'Shub-Niggurath',
    'Cthulhu',
    'Dagon',
    'Yig',
    'Tsathoggua',
    'Old Ones',
    'Deep Ones',
    'Shoggoth',
    #'Shoggoths',
    'Mi-Go',
    'night-gaunt',
    #'night-gaunts'
    # "Cor (Colour) do Espaço" # pensar no que fazer com esse
]

artefacts = [
    "Necronomicon",
    "Shining Trapezohedron"
]

places = [
    "R’lyeh",
    "Leng",
    "Yuggoth"
]

In [71]:
entities_df = pd.DataFrame() # creates an empty DataFrame to store entities
entities_df['name'] = entities_list

# verifying quantity of texts in which each entity appears
entities_df['number_of_texts'] = [ 
    sum([text_tools.search_name('data/silver/texts/'+file, entity) for file in biblio.files]) 
    for entity in entities_list ]

entities_df
entities_df.to_csv('data/gold/entities.csv')